In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, count

STEP 1: CREATE SPARK SESSION

In [ ]:
spark = SparkSession.builder \
    .appName("Practical8") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


STEP 2: CREATE SAMPLE SALES CSV DATA

In [ ]:
sales_data = [
    (1, "Laptop", "Electronics", 50000),
    (2, "Mobile", "Electronics", 25000),
    (3, "Chair", "Furniture", 5000),
    (4, "Table", "Furniture", 8000),
    (5, "Headphones", "Electronics", 3000),
    (6, "Pen", "Stationery", 100),
    (7, "Notebook", "Stationery", 200),
    (8, "Laptop", "Electronics", 50000),
    (8, "Laptop", "Electronics", 50000)
]

columns = [
    "Product_ID",
    "Product",
    "Category",
    "Sales"
]

sales_df = spark.createDataFrame(
    sales_data,
    columns
)

sales_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/sales_csv")

STEP 3: READ CSV DATASET

In [ ]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/content/sales_csv")

print("\n--- CSV Dataset ---")
df.show()


--- CSV Dataset ---
+----------+----------+-----------+-----+
|Product_ID|   Product|   Category|Sales|
+----------+----------+-----------+-----+
|         5|Headphones|Electronics| 3000|
|         6|       Pen| Stationery|  100|
|         7|  Notebook| Stationery|  200|
|         8|    Laptop|Electronics|50000|
|         8|    Laptop|Electronics|50000|
|         1|    Laptop|Electronics|50000|
|         2|    Mobile|Electronics|25000|
|         3|     Chair|  Furniture| 5000|
|         4|     Table|  Furniture| 8000|
+----------+----------+-----------+-----+



STEP 4: DISPLAY SCHEMA

In [ ]:
print("\n--- Dataset Schema ---")
df.printSchema()


--- Dataset Schema ---
root
 |-- Product_ID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sales: integer (nullable = true)



STEP 5: FILTERING

In [ ]:
print("\n--- Products with Sales Greater Than 5000 ---")

filtered_df = df.filter(
    col("Sales") > 5000
)

filtered_df.show()


--- Products with Sales Greater Than 5000 ---
+----------+-------+-----------+-----+
|Product_ID|Product|   Category|Sales|
+----------+-------+-----------+-----+
|         8| Laptop|Electronics|50000|
|         8| Laptop|Electronics|50000|
|         1| Laptop|Electronics|50000|
|         2| Mobile|Electronics|25000|
|         4|  Table|  Furniture| 8000|
+----------+-------+-----------+-----+



STEP 6: GROUPING

In [ ]:
print("\n--- Number of Products in Each Category ---")

grouped_df = df.groupBy(
    "Category"
).count()

grouped_df.show()


--- Number of Products in Each Category ---
+-----------+-----+
|   Category|count|
+-----------+-----+
| Stationery|    2|
|Electronics|    5|
|  Furniture|    2|
+-----------+-----+



STEP 7: AGGREGATION

In [ ]:
print("\n--- Total Sales by Category ---")

aggregation_df = df.groupBy(
    "Category"
).agg(
    sum("Sales").alias("Total_Sales"),
    count("Product_ID").alias("Number_of_Records")
)

aggregation_df.show()


--- Total Sales by Category ---
+-----------+-----------+-----------------+
|   Category|Total_Sales|Number_of_Records|
+-----------+-----------+-----------------+
| Stationery|        300|                2|
|Electronics|     178000|                5|
|  Furniture|      13000|                2|
+-----------+-----------+-----------------+



STEP 8: REMOVE DUPLICATE RECORDS

In [ ]:
print("\n--- Dataset After Removing Duplicates ---")

duplicate_removed_df = df.dropDuplicates()

duplicate_removed_df.show()

print(
    "Original record count:",
    df.count()
)

print(
    "Record count after removing duplicates:",
    duplicate_removed_df.count()
)


--- Dataset After Removing Duplicates ---
+----------+----------+-----------+-----+
|Product_ID|   Product|   Category|Sales|
+----------+----------+-----------+-----+
|         7|  Notebook| Stationery|  200|
|         5|Headphones|Electronics| 3000|
|         8|    Laptop|Electronics|50000|
|         6|       Pen| Stationery|  100|
|         4|     Table|  Furniture| 8000|
|         3|     Chair|  Furniture| 5000|
|         1|    Laptop|Electronics|50000|
|         2|    Mobile|Electronics|25000|
+----------+----------+-----------+-----+

Original record count: 9
Record count after removing duplicates: 8


STEP 9: CREATE SECOND DATASET FOR JOIN

In [ ]:
product_details = [
    (1, "Mumbai"),
    (2, "Pune"),
    (3, "Delhi"),
    (4, "Mumbai"),
    (5, "Pune"),
    (6, "Delhi"),
    (7, "Mumbai"),
    (8, "Mumbai")
]

location_columns = [
    "Product_ID",
    "Location"
]

location_df = spark.createDataFrame(
    product_details,
    location_columns
)

print("\n--- Product Location Dataset ---")
location_df.show()


--- Product Location Dataset ---
+----------+--------+
|Product_ID|Location|
+----------+--------+
|         1|  Mumbai|
|         2|    Pune|
|         3|   Delhi|
|         4|  Mumbai|
|         5|    Pune|
|         6|   Delhi|
|         7|  Mumbai|
|         8|  Mumbai|
+----------+--------+



STEP 10: JOIN TWO DATASETS

In [ ]:
print("\n--- Joined Dataset ---")

joined_df = duplicate_removed_df.join(
    location_df,
    on="Product_ID",
    how="inner"
)

joined_df.show()


--- Joined Dataset ---
+----------+----------+-----------+-----+--------+
|Product_ID|   Product|   Category|Sales|Location|
+----------+----------+-----------+-----+--------+
|         1|    Laptop|Electronics|50000|  Mumbai|
|         2|    Mobile|Electronics|25000|    Pune|
|         3|     Chair|  Furniture| 5000|   Delhi|
|         4|     Table|  Furniture| 8000|  Mumbai|
|         5|Headphones|Electronics| 3000|    Pune|
|         6|       Pen| Stationery|  100|   Delhi|
|         7|  Notebook| Stationery|  200|  Mumbai|
|         8|    Laptop|Electronics|50000|  Mumbai|
+----------+----------+-----------+-----+--------+



STEP 11: CALCULATE AVERAGE SALES BY CATEGORY

In [ ]:
print("\n--- Average Sales by Product Category ---")

average_sales_df = duplicate_removed_df.groupBy(
    "Category"
).agg(
    avg("Sales").alias("Average_Sales")
)

average_sales_df.show()


--- Average Sales by Product Category ---
+-----------+-------------+
|   Category|Average_Sales|
+-----------+-------------+
| Stationery|        150.0|
|Electronics|      32000.0|
|  Furniture|       6500.0|
+-----------+-------------+



STEP 12: STOP SPARK SESSION


In [ ]:
spark.stop()

print("\nPySpark Practical Completed Successfully!")


PySpark Practical Completed Successfully!
